# 017 Matched PCA–MS-AA Benchmark and PCA-Equivalent Dimensionality

This notebook provides a **targeted PCA control** for the MS-AA manuscript revision.

The previous version attempted to construct separate "spatial PCA" and "temporal PCA" orientations. That is not the comparison we want. Here PCA is treated as a conventional dimensionality-reduction benchmark:

\[
X^{(s)} \in \mathbb{R}^{T\times V}
\quad\longrightarrow\quad
Z^{(s)}_K \in \mathbb{R}^{T\times K},
\]

where PCA compresses the **700 HTFA node dimension**.

The notebook asks three focused questions:

1. **How does conventional PCA preserve cross-subject timepoint structure as dimensionality \(K\) changes?**
2. **How do those PCA curves compare with the full spatial- and temporal-MS-AA reconstruction curves?**
3. **For a sparse top-\(m\) MS-AA reconstruction, how many PCA dimensions are required to reach the same timepoint-identification accuracy?**

The third analysis gives an intuitive "PCA-equivalent dimensionality":

> e.g., top-5 spatial archetypes from a \(K=50\) AA decomposition preserve as much cross-subject temporal information as approximately \(q\) conventional PCA dimensions.

## Two PCA decoding modes are calculated

### A. Reconstruction-space PCA — primary matched control

PCA scores are projected back into the original \(T\times700\) HTFA space before timepoint matching. This matches the current MS-AA pipeline, which also reconstructs each subject into the original node space before decoding.

### B. Projected-space PCA — validation against prior PCA work

Timepoint matching is performed directly on the \(T\times K\) PCA scores. This mirrors the PCA temporal-decoding analysis used in the previously published compressibility work and provides a sanity check that the conventional PCA result behaves as expected.

For the direct **PCA vs MS-AA** comparison, use reconstruction-space PCA.


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

from pathlib import Path

# ----------------------------
# Data
# ----------------------------
CONDITIONS = ["intact", "word", "rest"]
TRIM_REST_BY = 100

PIEMAN_MAT_PATH = "data/pieman/raw/pieman_data.mat"
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"

# ----------------------------
# PCA fitting
# ----------------------------
# "across": one shared PCA basis fit using subjects/timepoints from
#           intact + word + rest. This is the primary matched control
#           for the across-condition MS-AA model.
#
# "within": separate PCA basis per condition, matching the design of
#           the prior PCA/compressibility paper.
PCA_FIT_SCOPE = "across"

# Conventional PCA dimensions. PCA operates on the 700 HTFA nodes.
# Dense sampling at low/intermediate K makes the AA equivalence
# calculation more precise.
PCA_K_VALUES = sorted(set([
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
    12, 14, 15, 18, 20, 25, 30, 35, 40, 45, 50,
    60, 70, 75, 80, 88, 100, 120, 126, 140, 150,
    175, 189, 200, 210, 225, 250, 252, 275, 300,
    350, 400, 450, 500, 550, 600, 650, 700
]))

PCA_SOLVER = "full"
PCA_WHITEN = False

# ----------------------------
# Decoder — matches notebook 002
# ----------------------------
NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42

# ----------------------------
# Existing AA outputs
# ----------------------------
AA_OUTPUT_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_across"

# If your filenames differ, edit these only.
AA_ANALYSIS_TYPES = ["spatial", "temporal"]

# ----------------------------
# Sparse AA values emphasized
# ----------------------------
AA_TOP_M_TO_COMPARE = [3, 5, 7, 10, 15]

# Optional specific K values for compact equivalence tables/plots.
# None means use every available AA K.
AA_K_FOCUS = {
    "spatial": [35, 48, 50, 70, 88, 100, 126, 140, 175, 200, 300, 700],
    "temporal": [10, 14, 15, 20, 23, 25, 30, 38, 45, 50, 75, 100, 200, 300],
}

# ----------------------------
# Outputs
# ----------------------------
OUTPUT_DIR = Path("pca_aa_equivalence_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_ROOT = Path("/Users/lowen/Desktop/papers/archetypes/figures")
FIG_DIR = FIG_ROOT / "017_matched_pca_aa_equivalence"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

print("PCA_FIT_SCOPE =", PCA_FIT_SCOPE)
print("Output directory:", OUTPUT_DIR.resolve())
print("Figure directory:", FIG_DIR)


In [ ]:
# ============================================================
# IMPORTS + PLOT HELPERS
# ============================================================

%matplotlib inline

import os
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA

plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COND_COLORS = {
    "intact": "purple",
    "word": "green",
    "rest": "black",
}

_fig_counter = 0

def _safe_name(name):
    name = str(name)
    return "".join(
        c if c.isalnum() or c in "_-." else "_"
        for c in name
    )[:180]

def save_current_fig(name):
    global _fig_counter
    if not SAVE_FIGS:
        return None

    _fig_counter += 1
    out = FIG_DIR / f"{_fig_counter:03d}_{_safe_name(name)}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out


## 1. Load exactly the same normalized HTFA data used by the MS-AA fitting notebook

This section intentionally mirrors notebook `001_fit_msaa_flexible`.


In [ ]:
# ============================================================
# DATA LOADING / NORMALIZATION
# mirrors notebook 001
# ============================================================

HT_NORMALIZE_AVAILABLE = False

try:
    spec = importlib.util.spec_from_file_location(
        "normalize",
        HYPERTOOLS_NORMALIZE_PATH,
    )
    ht = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ht)
    HT_NORMALIZE_AVAILABLE = True
except Exception:
    ht = None

def to_float_array(x):
    return np.array(x, dtype=float)

def array_cutter(data_list, cut_length):
    out = [to_float_array(x).copy() for x in data_list]

    if cut_length <= 0:
        return out

    return [x[:-cut_length] for x in out]

def coerce_TxV(x):
    x = to_float_array(x)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D array, got {x.shape}")

    T0, V0 = x.shape

    # Same convention as notebook 001
    if V0 < T0:
        x = x.T

    x = x - x.mean(axis=0, keepdims=True)

    return x.astype(np.float32, copy=False)

def normalize_across_fallback(subject_list, eps=1e-8):
    subj = [coerce_TxV(x) for x in subject_list]

    stacked = np.vstack(subj)

    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + eps

    return [
        ((x - mu) / sd).astype(np.float32, copy=False)
        for x in subj
    ]

def normalize_subject_list(subject_list):
    subj = [coerce_TxV(x) for x in subject_list]

    if HT_NORMALIZE_AVAILABLE:
        try:
            return ht.normalize(
                subj,
                normalize="across",
            )
        except Exception:
            return normalize_across_fallback(subj)

    return normalize_across_fallback(subj)

def load_condition_data():
    pieman_data = loadmat(PIEMAN_MAT_PATH)

    data = []
    conds = []

    for c in CONDITIONS:
        next_data = list(
            map(
                lambda i: pieman_data[c][:, i][0],
                np.arange(pieman_data[c].shape[1]),
            )
        )

        data.extend(next_data)
        conds.extend([c] * len(next_data))

    conds_array = np.asarray(conds)

    condition_data = {}

    for c in CONDITIONS:
        cur = [
            data[i]
            for i in np.where(conds_array == c)[0]
        ]

        if c == "rest" and TRIM_REST_BY > 0:
            cur = array_cutter(
                cur,
                TRIM_REST_BY,
            )

        condition_data[c] = normalize_subject_list(cur)

    return condition_data

condition_data = load_condition_data()

print("Condition counts and shapes:")

for c in CONDITIONS:
    print(
        c,
        "N =", len(condition_data[c]),
        "shape =", np.asarray(condition_data[c][0]).shape,
    )

T = int(np.asarray(condition_data["intact"][0]).shape[0])
V = int(np.asarray(condition_data["intact"][0]).shape[1])

print("\nT =", T)
print("V =", V)
print("Chance =", 1.0 / T)

assert V == 700, (
    f"Expected V=700 HTFA nodes, got V={V}. "
    "Check data orientation before proceeding."
)

assert max(PCA_K_VALUES) <= V


## 2. Fit conventional PCA on the 700-node dimension

For the primary `PCA_FIT_SCOPE="across"` analysis, one shared PCA basis is estimated from:

\[
\begin{bmatrix}
X^{(1)}_{\mathrm{intact}}\\
\vdots\\
X^{(N)}_{\mathrm{intact}}\\
X^{(1)}_{\mathrm{word}}\\
\vdots\\
X^{(N)}_{\mathrm{rest}}
\end{bmatrix}
\in \mathbb{R}^{(N_{\mathrm{total}}T)\times700}.
\]

Thus **each row is a timepoint** and **each PCA feature is one of the 700 HTFA nodes**.

This is the conventional PCA compression of spatial brain activity used in the earlier compressibility analysis. The difference is that the primary model here uses one across-condition basis so that its fitting scope matches the current across-condition MS-AA analysis.


In [ ]:
# ============================================================
# FIT CONVENTIONAL NODE-SPACE PCA
# ============================================================

def fit_conventional_pca(condition_data, fit_scope="across"):
    models = {}

    if fit_scope == "across":

        pooled = np.vstack([
            X
            for c in CONDITIONS
            for X in condition_data[c]
        ])

        print("Across-condition PCA matrix:", pooled.shape)
        print(
            "Interpretation:",
            pooled.shape[0],
            "timepoint observations x",
            pooled.shape[1],
            "HTFA-node features",
        )

        model = PCA(
            n_components=V,
            svd_solver=PCA_SOLVER,
            whiten=PCA_WHITEN,
            random_state=RNG_SEED,
        )

        model.fit(pooled)
        models["across"] = model

    elif fit_scope == "within":

        for c in CONDITIONS:

            pooled = np.vstack(condition_data[c])

            print(
                c,
                "PCA matrix:",
                pooled.shape,
            )

            model = PCA(
                n_components=V,
                svd_solver=PCA_SOLVER,
                whiten=PCA_WHITEN,
                random_state=RNG_SEED,
            )

            model.fit(pooled)
            models[c] = model

    else:
        raise ValueError(
            "PCA_FIT_SCOPE must be 'across' or 'within'"
        )

    return models

pca_models = fit_conventional_pca(
    condition_data,
    fit_scope=PCA_FIT_SCOPE,
)

def get_pca_model(condition):
    if PCA_FIT_SCOPE == "across":
        return pca_models["across"]
    return pca_models[condition]

print("\nExplained variance sanity checks:")

for c in CONDITIONS:
    model = get_pca_model(c)

    print(
        c,
        "10 PCs:",
        model.explained_variance_ratio_[:10].sum(),
        "| 50 PCs:",
        model.explained_variance_ratio_[:50].sum(),
        "| 100 PCs:",
        model.explained_variance_ratio_[:100].sum(),
        "| 700 PCs:",
        model.explained_variance_ratio_[:700].sum(),
    )

    if PCA_FIT_SCOPE == "across":
        break


## 3. PCA transformations

We calculate two representations for each \(K\).

### Projected PCA

\[
Z_K = (X-\mu)P_K^\top
\]

with shape \(T\times K\).

### Reconstructed PCA

\[
\hat X_K = Z_KP_K+\mu
\]

with shape \(T\times700\), followed by the **same within-subject node-wise temporal z-scoring used for MS-AA reconstructions**.

The reconstructed version is the direct matched control for MS-AA.


In [ ]:
# ============================================================
# PCA TRANSFORM / RECONSTRUCTION
# ============================================================

def normalize_reconstruction_like_msaa(Xhat):
    Xhat = np.asarray(Xhat, dtype=float)

    return (
        Xhat - Xhat.mean(axis=0, keepdims=True)
    ) / (
        Xhat.std(axis=0, keepdims=True) + 1e-8
    )

def pca_project_subject(X, model, K):
    """
    Return T x K conventional PCA scores.
    """
    X = np.asarray(X, dtype=float)

    Z_full = model.transform(X)

    return Z_full[:, :K]

def pca_reconstruct_subject(X, model, K):
    """
    Reconstruct into T x V using the first K conventional PCs.
    """
    X = np.asarray(X, dtype=float)

    Z = model.transform(X)[:, :K]
    P = model.components_[:K, :]

    Xhat = Z @ P + model.mean_

    return normalize_reconstruction_like_msaa(Xhat)

def build_pca_stack(
    subjects,
    model,
    K,
    decode_space="reconstruction",
):
    if decode_space == "reconstruction":
        return np.stack([
            pca_reconstruct_subject(X, model, K)
            for X in subjects
        ], axis=0)

    if decode_space == "projected":
        return np.stack([
            pca_project_subject(X, model, K)
            for X in subjects
        ], axis=0)

    raise ValueError(
        "decode_space must be 'reconstruction' or 'projected'"
    )


## 4. Cross-subject timepoint matching

This is copied from notebook `002_decode_msaa`: subjects are repeatedly divided into two groups, group-average representations are correlated across all pairs of timepoints, and the highest-correlation timepoint is selected.

Chance exact identification is \(1/T\).


In [ ]:
# ============================================================
# DECODER
# mirrors notebook 002
# ============================================================

def decoder(corrs):
    out = pd.DataFrame({
        "rank": [0.0],
        "accuracy": [0.0],
        "error": [0.0],
    })

    T_local = corrs.shape[0]

    for t in range(T_local):

        decoded_ind = int(
            np.argmax(corrs[t, :])
        )

        out.loc[0, "error"] += (
            np.mean(np.abs(decoded_ind - t))
            / T_local
        )

        out.loc[0, "accuracy"] += (
            decoded_ind == t
        )

        out.loc[0, "rank"] += np.mean(
            (
                corrs[t, :]
                <= corrs[t, t]
            ).astype(int)
        )

    out["error"] /= T_local
    out["accuracy"] /= T_local
    out["rank"] /= T_local

    return out

def get_xval_assignments(
    ndata,
    nfolds,
    rng=None,
):
    rng = (
        np.random.default_rng()
        if rng is None
        else rng
    )

    group_assignments = np.zeros(
        ndata,
        dtype=int,
    )

    groupsize = int(
        np.ceil(ndata / nfolds)
    )

    for i in range(1, nfolds):

        inds = np.arange(
            i * groupsize,
            min(
                (i + 1) * groupsize,
                ndata,
            ),
        )

        group_assignments[inds] = i

    rng.shuffle(group_assignments)

    return group_assignments

def run_timepoint_decoding(
    representation_stack,
    nfolds=NFOLDS_DECODE,
    nreps=NREPS_DECODE,
    seed=RNG_SEED,
):
    N, T_local, D = representation_stack.shape

    rng = np.random.default_rng(seed)

    rows = []

    for rep in range(nreps):

        fold_ids = get_xval_assignments(
            N,
            nfolds,
            rng=rng,
        )

        for fold in range(nfolds):

            in_mask = (
                fold_ids == fold
            )

            out_mask = ~in_mask

            in_mean = representation_stack[
                in_mask
            ].mean(axis=0)

            out_mean = representation_stack[
                out_mask
            ].mean(axis=0)

            corrs = (
                1.0
                - cdist(
                    in_mean,
                    out_mean,
                    metric="correlation",
                )
            )

            corrs = np.nan_to_num(
                corrs,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            res = decoder(corrs)

            res["rep"] = rep
            res["fold"] = fold

            rows.append(res)

    return pd.concat(
        rows,
        ignore_index=True,
    )

def summarize_decoding_runs(dec_df):
    return {
        "mean": float(
            dec_df["accuracy"].mean()
        ),
        "err": float(
            dec_df["accuracy"].std()
            / np.sqrt(max(len(dec_df), 1))
        ),
        "std": float(
            dec_df["accuracy"].std()
        ),
        "count": int(
            len(dec_df)
        ),
        "rank_mean": float(
            dec_df["rank"].mean()
        ),
        "error_mean": float(
            dec_df["error"].mean()
        ),
    }


## 5. Raw 700-node HTFA reference

This is not a PCA model. It establishes the performance of the original normalized HTFA representation under the same timepoint-matching procedure.


In [ ]:
# ============================================================
# RAW HTFA BASELINE
# ============================================================

raw_rows = []

for condition in CONDITIONS:

    raw_stack = np.stack([
        normalize_reconstruction_like_msaa(X)
        for X in condition_data[condition]
    ], axis=0)

    dec_df = run_timepoint_decoding(
        raw_stack,
    )

    summary = summarize_decoding_runs(
        dec_df
    )

    raw_rows.append({
        "method": "raw_htfa",
        "condition": condition,
        "K": V,
        "component_ratio": 1.0,
        "chance": 1.0 / T,
        **summary,
    })

raw_summary_df = pd.DataFrame(
    raw_rows
)

display(raw_summary_df)

raw_summary_df.to_csv(
    OUTPUT_DIR / "raw_htfa_summary.csv",
    index=False,
)


## 6. PCA decoding curves

This computes both projected- and reconstructed-space PCA curves using the **same fitted PCA basis** and the same subject splits/decoder settings.

The projected-space curve is a methodological sanity check against the prior PCA result.

The reconstructed-space curve is the direct matched comparator for MS-AA.


In [ ]:
# ============================================================
# PCA CURVES
# ============================================================

pca_rows = []

for condition in CONDITIONS:

    model = get_pca_model(condition)
    subjects = condition_data[condition]

    for K in PCA_K_VALUES:

        if K > model.n_components_:
            continue

        print(
            condition,
            "K =", K,
        )

        for decode_space in [
            "projected",
            "reconstruction",
        ]:

            stack = build_pca_stack(
                subjects,
                model,
                K,
                decode_space=decode_space,
            )

            dec_df = run_timepoint_decoding(
                stack,
            )

            summary = summarize_decoding_runs(
                dec_df
            )

            pca_rows.append({
                "method": "PCA",
                "fit_scope": PCA_FIT_SCOPE,
                "decode_space": decode_space,
                "condition": condition,
                "K": int(K),
                "component_ratio": K / V,
                "variance_explained": float(
                    model.explained_variance_ratio_[:K].sum()
                ),
                "chance": 1.0 / T,
                **summary,
            })

pca_summary_df = pd.DataFrame(
    pca_rows
)

pca_summary_df.to_csv(
    OUTPUT_DIR / "pca_summary.csv",
    index=False,
)

display(
    pca_summary_df.head(12)
)


## 7. PCA sanity-check plots

In [ ]:
# ============================================================
# PCA SANITY-CHECK PLOTS
# ============================================================

def plot_pca_decode_modes(condition):

    sub = pca_summary_df[
        pca_summary_df["condition"]
        == condition
    ].copy()

    fig, ax = plt.subplots(
        figsize=(7.5, 4.8)
    )

    for decode_space, linestyle in [
        ("projected", "--"),
        ("reconstruction", "-"),
    ]:

        s = sub[
            sub["decode_space"]
            == decode_space
        ].sort_values("K")

        ax.errorbar(
            s["K"],
            s["mean"],
            yerr=s["err"],
            linestyle=linestyle,
            marker="o",
            markersize=3,
            capsize=2,
            linewidth=1.8,
            label=decode_space,
        )

    raw = raw_summary_df[
        raw_summary_df["condition"]
        == condition
    ]

    if len(raw):
        ax.axhline(
            float(raw.iloc[0]["mean"]),
            linestyle=":",
            linewidth=1.5,
            label="raw HTFA",
        )

    ax.axhline(
        1.0 / T,
        color="gray",
        linestyle=":",
        linewidth=1,
        label="chance",
    )

    ax.set_xlabel(
        "Number of PCA components"
    )

    ax.set_ylabel(
        "Cross-subject timepoint-identification accuracy"
    )

    ax.set_title(
        f"PCA sanity check | {condition}"
    )

    ax.legend(
        frameon=False
    )

    plt.tight_layout()

    save_current_fig(
        f"PCA_decode_modes_{condition}"
    )

    plt.show()
    plt.close()

for condition in CONDITIONS:
    plot_pca_decode_modes(condition)


In [ ]:
# ============================================================
# ALL CONDITIONS — PRIMARY RECONSTRUCTION-SPACE PCA CURVE
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.0, 5.0)
)

for condition in CONDITIONS:

    s = pca_summary_df[
        (pca_summary_df["condition"] == condition) &
        (pca_summary_df["decode_space"] == "reconstruction")
    ].sort_values("K")

    ax.errorbar(
        s["K"],
        s["mean"],
        yerr=s["err"],
        color=COND_COLORS[condition],
        marker="o",
        markersize=3,
        capsize=2,
        linewidth=1.8,
        label=condition,
    )

ax.axhline(
    1.0 / T,
    color="gray",
    linestyle=":",
    linewidth=1,
    label="chance",
)

ax.set_xlabel(
    "Number of PCA components"
)

ax.set_ylabel(
    "Cross-subject timepoint-identification accuracy"
)

ax.set_title(
    f"Conventional PCA compression | {PCA_FIT_SCOPE}-condition basis"
)

ax.legend(
    frameon=False
)

plt.tight_layout()

save_current_fig(
    "PCA_reconstruction_all_conditions"
)

plt.show()
plt.close()


## 8. Load the existing spatial and temporal MS-AA results

This does **not rerun MS-AA**. It loads `full_summary.csv` and `topm_summary.csv` from notebook 002 outputs.


In [ ]:
# ============================================================
# LOAD AA SUMMARIES
# ============================================================

aa_full_parts = []
aa_topm_parts = []

for analysis_type in AA_ANALYSIS_TYPES:

    d = Path(
        AA_OUTPUT_TEMPLATE.format(
            analysis_type=analysis_type
        )
    )

    full_path = d / "full_summary.csv"
    topm_path = d / "topm_summary.csv"

    if full_path.exists():

        x = pd.read_csv(
            full_path
        )

        if "analysis_type" not in x.columns:
            x["analysis_type"] = analysis_type

        x["method"] = "MS-AA"

        # Use the native dimensionality relevant to each AA orientation.
        D = (
            V
            if analysis_type == "spatial"
            else T
        )

        x["component_ratio"] = (
            x["K"] / D
        )

        aa_full_parts.append(x)

        print(
            "Loaded:",
            full_path,
        )

    else:
        print(
            "Missing:",
            full_path,
        )

    if topm_path.exists():

        x = pd.read_csv(
            topm_path
        )

        if "analysis_type" not in x.columns:
            x["analysis_type"] = analysis_type

        x["method"] = "MS-AA"

        D = (
            V
            if analysis_type == "spatial"
            else T
        )

        x["component_ratio"] = (
            x["K"] / D
        )

        aa_topm_parts.append(x)

        print(
            "Loaded:",
            topm_path,
        )

    else:
        print(
            "Missing:",
            topm_path,
        )

aa_full_df = (
    pd.concat(
        aa_full_parts,
        ignore_index=True,
    )
    if aa_full_parts
    else pd.DataFrame()
)

aa_topm_df = (
    pd.concat(
        aa_topm_parts,
        ignore_index=True,
    )
    if aa_topm_parts
    else pd.DataFrame()
)

print(
    "AA full rows:",
    len(aa_full_df),
)

print(
    "AA top-m rows:",
    len(aa_topm_df),
)

display(
    aa_full_df.head()
    if len(aa_full_df)
    else pd.DataFrame()
)


## 9. Direct full-model comparison

PCA uses \(K/700\) on the x-axis because PCA compresses the 700-node HTFA representation.

Spatial MS-AA also uses \(K/700\).

Temporal MS-AA uses \(K/T\).

This plot is therefore a **normalized representational-ratio comparison**, not a claim that the latent coordinates are mathematically identical across methods.


In [ ]:
# ============================================================
# PCA VS FULL MS-AA
# ============================================================

def plot_pca_vs_aa_full(condition):

    fig, ax = plt.subplots(
        figsize=(8.2, 5.1)
    )

    # Primary PCA — reconstruction-space
    pca = pca_summary_df[
        (pca_summary_df["condition"] == condition) &
        (pca_summary_df["decode_space"] == "reconstruction")
    ].sort_values("component_ratio")

    ax.errorbar(
        pca["component_ratio"],
        pca["mean"],
        yerr=pca["err"],
        marker="o",
        markersize=3,
        capsize=2,
        linewidth=2.0,
        linestyle="-.",
        label="PCA",
    )

    # Spatial / temporal AA
    if len(aa_full_df):

        for analysis_type, linestyle in [
            ("spatial", "-"),
            ("temporal", "--"),
        ]:

            aa = aa_full_df[
                (aa_full_df["analysis_type"].astype(str) == analysis_type) &
                (aa_full_df["condition"].astype(str) == condition)
            ].sort_values("component_ratio")

            if len(aa) == 0:
                continue

            yerr = (
                aa["err"]
                if "err" in aa.columns
                else None
            )

            ax.errorbar(
                aa["component_ratio"],
                aa["mean"],
                yerr=yerr,
                marker="o",
                markersize=3,
                capsize=2,
                linewidth=2.0,
                linestyle=linestyle,
                label=f"{analysis_type} MS-AA",
            )

    raw = raw_summary_df[
        raw_summary_df["condition"]
        == condition
    ]

    if len(raw):
        ax.axhline(
            float(raw.iloc[0]["mean"]),
            color="gray",
            linestyle=":",
            linewidth=1.5,
            label="raw HTFA",
        )

    ax.axhline(
        1.0 / T,
        color="black",
        linestyle=":",
        linewidth=1,
        alpha=0.7,
        label="chance",
    )

    ax.set_xlabel(
        "Normalized representational ratio"
    )

    ax.set_ylabel(
        "Cross-subject timepoint-identification accuracy"
    )

    ax.set_title(
        f"PCA vs MS-AA | {condition}"
    )

    ax.legend(
        frameon=False
    )

    plt.tight_layout()

    save_current_fig(
        f"PCA_vs_MSAA_full_{condition}"
    )

    plt.show()
    plt.close()

for condition in CONDITIONS:
    plot_pca_vs_aa_full(condition)


## 10. PCA-equivalent dimensionality of sparse AA

For each AA top-\(m\) configuration, we find the PCA dimensionality whose **reconstruction-space** accuracy is closest to the AA accuracy.

We report two quantities:

### Closest PCA dimensionality

\[
K_{\mathrm{PCA,closest}}
=
\arg\min_q
\left|
A_{\mathrm{PCA}}(q)-A_{\mathrm{AA}}(K,m)
\right|.
\]

### Minimum PCA dimensionality reaching AA performance

\[
K_{\mathrm{PCA,min}}
=
\min
\left\{
q:
A_{\mathrm{PCA}}(q)\geq
A_{\mathrm{AA}}(K,m)
\right\}.
\]

The second is the more intuitive "how many conventional PCA dimensions are needed to reach this AA performance?" measure.

Because empirical PCA curves need not be perfectly monotonic, we also compute a **monotonic envelope** using the cumulative maximum of PCA accuracy across increasing \(K\). The minimum-reaching metric is calculated from that envelope by default.


In [ ]:
# ============================================================
# PCA-EQUIVALENT DIMENSIONALITY
# ============================================================

def prepare_pca_curve_for_equivalence(
    condition,
    use_monotonic_envelope=True,
):

    p = pca_summary_df[
        (pca_summary_df["condition"] == condition) &
        (pca_summary_df["decode_space"] == "reconstruction")
    ].copy()

    p = p.sort_values(
        "K"
    ).reset_index(drop=True)

    if use_monotonic_envelope:
        p["equivalence_accuracy"] = (
            p["mean"].cummax()
        )
    else:
        p["equivalence_accuracy"] = (
            p["mean"]
        )

    return p

def pca_equivalent_for_accuracy(
    target_accuracy,
    condition,
    use_monotonic_envelope=True,
):

    p = prepare_pca_curve_for_equivalence(
        condition,
        use_monotonic_envelope=use_monotonic_envelope,
    )

    # Closest empirical point
    closest_idx = (
        p["mean"] - target_accuracy
    ).abs().idxmin()

    closest = p.loc[
        closest_idx
    ]

    # First K reaching target on monotonic envelope
    reaching = p[
        p["equivalence_accuracy"]
        >= target_accuracy
    ]

    if len(reaching):

        minimum = reaching.iloc[0]

        k_min = int(
            minimum["K"]
        )

        acc_min = float(
            minimum["equivalence_accuracy"]
        )

    else:

        k_min = np.nan
        acc_min = np.nan

    return {
        "pca_K_closest": int(
            closest["K"]
        ),
        "pca_accuracy_closest": float(
            closest["mean"]
        ),
        "pca_abs_difference": float(
            abs(
                closest["mean"]
                - target_accuracy
            )
        ),
        "pca_K_min_reaching": k_min,
        "pca_accuracy_at_min_reaching": acc_min,
    }

equivalence_rows = []

if len(aa_topm_df):

    aa_use = aa_topm_df[
        aa_topm_df["top_m"].astype(int).isin(
            AA_TOP_M_TO_COMPARE
        )
    ].copy()

    for _, r in aa_use.iterrows():

        condition = str(
            r["condition"]
        )

        analysis_type = str(
            r["analysis_type"]
        )

        K_aa = int(
            r["K"]
        )

        top_m = int(
            r["top_m"]
        )

        aa_accuracy = float(
            r["mean"]
        )

        eq = pca_equivalent_for_accuracy(
            aa_accuracy,
            condition,
            use_monotonic_envelope=True,
        )

        equivalence_rows.append({
            "analysis_type": analysis_type,
            "condition": condition,
            "AA_K": K_aa,
            "AA_component_ratio": float(
                r["component_ratio"]
            ),
            "top_m": top_m,
            "AA_accuracy": aa_accuracy,
            "AA_err": (
                float(r["err"])
                if "err" in r.index
                else np.nan
            ),
            **eq,
        })

aa_pca_equivalence_df = pd.DataFrame(
    equivalence_rows
)

aa_pca_equivalence_df.to_csv(
    OUTPUT_DIR / "aa_pca_equivalent_dimensions.csv",
    index=False,
)

display(
    aa_pca_equivalence_df.head(20)
)


## 11. Inspect the manuscript-relevant spatial sparse regime

This table is intentionally focused on the AA configurations you have been discussing, rather than selecting a new "optimal" \(K\) from the PCA comparison.


In [ ]:
# ============================================================
# FOCUSED EQUIVALENCE TABLE
# ============================================================

focus_parts = []

for analysis_type in AA_ANALYSIS_TYPES:

    k_focus = AA_K_FOCUS.get(
        analysis_type,
        None,
    )

    s = aa_pca_equivalence_df[
        aa_pca_equivalence_df["analysis_type"]
        == analysis_type
    ].copy()

    if k_focus is not None:
        s = s[
            s["AA_K"].isin(k_focus)
        ]

    focus_parts.append(s)

focus_equivalence_df = pd.concat(
    focus_parts,
    ignore_index=True,
)

focus_equivalence_df = focus_equivalence_df.sort_values([
    "analysis_type",
    "condition",
    "AA_K",
    "top_m",
])

display(
    focus_equivalence_df[[
        "analysis_type",
        "condition",
        "AA_K",
        "top_m",
        "AA_accuracy",
        "pca_K_min_reaching",
        "pca_K_closest",
        "pca_accuracy_closest",
        "pca_abs_difference",
    ]]
)

focus_equivalence_df.to_csv(
    OUTPUT_DIR / "aa_pca_equivalence_focus.csv",
    index=False,
)


## 12. Figure: AA sparse accuracy and its PCA-equivalent dimensionality

The left y-axis shows the AA top-\(m\) accuracy. The corresponding table/labels indicate how many PCA dimensions are required to reach the same accuracy.

For the main manuscript, I would probably show only **intact** and perhaps **word**, with a small set of spatial \(K\) values selected *a priori* from the existing AA analyses rather than from this PCA benchmark.


In [ ]:
# ============================================================
# EQUIVALENT DIMENSIONALITY PLOT
# ============================================================

def plot_equivalent_dimensions(
    analysis_type="spatial",
    condition="intact",
    top_m=5,
    k_values=None,
):

    s = aa_pca_equivalence_df[
        (aa_pca_equivalence_df["analysis_type"] == analysis_type) &
        (aa_pca_equivalence_df["condition"] == condition) &
        (aa_pca_equivalence_df["top_m"] == top_m)
    ].copy()

    if k_values is not None:
        s = s[
            s["AA_K"].isin(k_values)
        ]

    s = s.sort_values(
        "AA_K"
    )

    if len(s) == 0:
        print(
            "No rows:",
            analysis_type,
            condition,
            top_m,
        )
        return

    fig, ax = plt.subplots(
        figsize=(8.0, 5.0)
    )

    ax.plot(
        s["AA_K"],
        s["AA_accuracy"],
        marker="o",
        linewidth=2,
        label=f"{analysis_type} MS-AA top-{top_m}",
    )

    for _, r in s.iterrows():

        k_eq = r[
            "pca_K_min_reaching"
        ]

        label = (
            "PCA > max"
            if pd.isna(k_eq)
            else f"PCA≈{int(k_eq)}"
        )

        ax.annotate(
            label,
            (
                r["AA_K"],
                r["AA_accuracy"],
            ),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    ax.axhline(
        1.0 / T,
        color="gray",
        linestyle=":",
        linewidth=1,
        label="chance",
    )

    ax.set_xlabel(
        "MS-AA model order K"
    )

    ax.set_ylabel(
        "Cross-subject timepoint-identification accuracy"
    )

    ax.set_title(
        f"{condition} | {analysis_type} AA top-{top_m}\n"
        "labels = minimum PCA dimensions reaching AA accuracy"
    )

    ax.legend(
        frameon=False
    )

    plt.tight_layout()

    save_current_fig(
        f"PCA_equivalent_{analysis_type}_{condition}_top{top_m}"
    )

    plt.show()
    plt.close()

for condition in ["intact", "word"]:

    plot_equivalent_dimensions(
        analysis_type="spatial",
        condition=condition,
        top_m=5,
        k_values=AA_K_FOCUS["spatial"],
    )

    plot_equivalent_dimensions(
        analysis_type="spatial",
        condition=condition,
        top_m=10,
        k_values=AA_K_FOCUS["spatial"],
    )


## 13. Figure: Sparse AA points overlaid on the PCA compression curve

This is probably the most intuitive reviewer figure.

- Line = conventional PCA reconstruction-space decoding across number of retained PCs.
- Points = selected MS-AA top-\(m\) reconstructions.
- The horizontal position of an AA point is **the number of archetypes actually retained (\(m\))**, not the parent model order \(K\).
- Point labels retain the parent AA \(K\), because the same \(m\) archetypes can carry different information depending on the decomposition from which they came.

This directly illustrates a key distinction:

> PCA is nested and ordered by variance, whereas a top-\(m\) AA subset is drawn from a \(K\)-dependent archetypal geometry.


In [ ]:
# ============================================================
# PCA CURVE + SPARSE AA POINTS
# ============================================================

def plot_pca_curve_with_sparse_aa(
    condition="intact",
    analysis_type="spatial",
    top_m_values=(3, 5, 10),
    k_values=None,
):

    pca = pca_summary_df[
        (pca_summary_df["condition"] == condition) &
        (pca_summary_df["decode_space"] == "reconstruction")
    ].sort_values("K")

    fig, ax = plt.subplots(
        figsize=(8.3, 5.2)
    )

    ax.plot(
        pca["K"],
        pca["mean"],
        linewidth=2.2,
        label="PCA",
    )

    aa = aa_topm_df[
        (aa_topm_df["analysis_type"].astype(str) == analysis_type) &
        (aa_topm_df["condition"].astype(str) == condition) &
        (aa_topm_df["top_m"].astype(int).isin(top_m_values))
    ].copy()

    if k_values is not None:
        aa = aa[
            aa["K"].astype(int).isin(
                k_values
            )
        ]

    marker_map = {
        3: "o",
        5: "s",
        7: "^",
        10: "D",
        15: "P",
    }

    for top_m in top_m_values:

        s = aa[
            aa["top_m"].astype(int)
            == int(top_m)
        ].copy()

        if len(s) == 0:
            continue

        # small horizontal jitter by parent K so overlapping m values remain visible
        k_parent = s["K"].astype(float).to_numpy()

        if len(k_parent) > 1:
            ranks = pd.Series(
                k_parent
            ).rank(
                method="first"
            ).to_numpy()

            jitter = (
                ranks
                - ranks.mean()
            ) * 0.10

        else:
            jitter = np.zeros(
                len(s)
            )

        x = (
            float(top_m)
            + jitter
        )

        ax.scatter(
            x,
            s["mean"],
            s=65,
            marker=marker_map.get(
                int(top_m),
                "o",
            ),
            edgecolor="black",
            linewidth=0.5,
            label=f"MS-AA top-{top_m}",
            zorder=5,
        )

        for xi, (_, r) in zip(
            x,
            s.iterrows(),
        ):

            ax.annotate(
                f"K={int(r['K'])}",
                (
                    xi,
                    r["mean"],
                ),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=7,
            )

    ax.axhline(
        1.0 / T,
        color="gray",
        linestyle=":",
        linewidth=1,
        label="chance",
    )

    ax.set_xlabel(
        "Retained dimensions / retained archetypes"
    )

    ax.set_ylabel(
        "Cross-subject timepoint-identification accuracy"
    )

    ax.set_title(
        f"{condition} | conventional PCA vs sparse {analysis_type} MS-AA"
    )

    # Zoom to sparse range while still showing enough PCA curve
    max_topm = max(top_m_values)
    ax.set_xlim(
        0,
        max(50, max_topm + 10),
    )

    ax.legend(
        frameon=False,
        fontsize=9,
    )

    plt.tight_layout()

    save_current_fig(
        f"PCA_curve_sparse_{analysis_type}_{condition}"
    )

    plt.show()
    plt.close()

for condition in ["intact", "word"]:

    plot_pca_curve_with_sparse_aa(
        condition=condition,
        analysis_type="spatial",
        top_m_values=(3, 5, 10),
        k_values=[48, 50, 70, 88, 100],
    )


## 14. A more conservative comparison: AA performance vs PCA performance at the same number of retained dimensions

This answers a slightly different question from PCA-equivalent dimensionality:

> If both methods retain exactly \(m\) latent elements, how much timepoint information is preserved?

For PCA, retain the first \(m\) variance-maximizing PCs.

For AA, use the current top-\(m\) decoding-selected subset.

**Important:** until AA ranking is made held-out, this comparison should be described as descriptive because the AA subset benefits from decoding-based selection whereas PCA's first \(m\) PCs are selected solely by explained variance.


In [ ]:
# ============================================================
# SAME-m COMPARISON
# ============================================================

same_m_rows = []

if len(aa_topm_df):

    for _, r in aa_topm_df.iterrows():

        top_m = int(
            r["top_m"]
        )

        condition = str(
            r["condition"]
        )

        if top_m not in PCA_K_VALUES:
            continue

        p = pca_summary_df[
            (pca_summary_df["condition"] == condition) &
            (pca_summary_df["decode_space"] == "reconstruction") &
            (pca_summary_df["K"] == top_m)
        ]

        if len(p) == 0:
            continue

        p = p.iloc[0]

        same_m_rows.append({
            "analysis_type": str(
                r["analysis_type"]
            ),
            "condition": condition,
            "AA_K": int(
                r["K"]
            ),
            "top_m": top_m,
            "AA_accuracy": float(
                r["mean"]
            ),
            "PCA_K": top_m,
            "PCA_accuracy": float(
                p["mean"]
            ),
            "AA_minus_PCA_same_m": float(
                r["mean"] - p["mean"]
            ),
        })

same_m_df = pd.DataFrame(
    same_m_rows
)

same_m_df.to_csv(
    OUTPUT_DIR / "aa_vs_pca_same_m.csv",
    index=False,
)

display(
    same_m_df.head(20)
)


## 15. Final compact tables for the revision

These tables intentionally avoid declaring that PCA or AA is "better." They summarize the empirical comparison needed for the reviewer response.


In [ ]:
# ============================================================
# COMPACT REVIEWER TABLES
# ============================================================

# Best PCA accuracy by condition
pca_best_rows = []

for condition in CONDITIONS:

    p = pca_summary_df[
        (pca_summary_df["condition"] == condition) &
        (pca_summary_df["decode_space"] == "reconstruction")
    ].copy()

    best = p.loc[
        p["mean"].idxmax()
    ]

    raw = raw_summary_df[
        raw_summary_df["condition"]
        == condition
    ].iloc[0]

    pca_best_rows.append({
        "condition": condition,
        "best_PCA_K": int(
            best["K"]
        ),
        "best_PCA_accuracy": float(
            best["mean"]
        ),
        "raw_HTFA_accuracy": float(
            raw["mean"]
        ),
        "PCA_minus_raw": float(
            best["mean"]
            - raw["mean"]
        ),
    })

pca_best_df = pd.DataFrame(
    pca_best_rows
)

print("Best PCA vs raw HTFA")
display(pca_best_df)

# Focused spatial AA equivalence table
spatial_focus = focus_equivalence_df[
    focus_equivalence_df["analysis_type"]
    == "spatial"
].copy()

spatial_focus = spatial_focus[
    spatial_focus["condition"].isin(
        ["intact", "word"]
    )
]

spatial_focus = spatial_focus[
    spatial_focus["top_m"].isin(
        [3, 5, 10]
    )
]

print("\nFocused spatial-AA to PCA equivalence")
display(
    spatial_focus[[
        "condition",
        "AA_K",
        "top_m",
        "AA_accuracy",
        "pca_K_min_reaching",
        "pca_K_closest",
        "pca_accuracy_closest",
    ]].sort_values([
        "condition",
        "AA_K",
        "top_m",
    ])
)

pca_best_df.to_csv(
    OUTPUT_DIR / "reviewer_pca_best_vs_raw.csv",
    index=False,
)

spatial_focus.to_csv(
    OUTPUT_DIR / "reviewer_spatial_AA_PCA_equivalence.csv",
    index=False,
)


# How to interpret the output

## First sanity check

Before looking at AA, inspect the PCA-only plots.

The **projected-space PCA curve** should broadly resemble the conventional compression result from the prior PCA analysis: performance should generally increase as more PCs are retained, with intact showing the strongest cross-subject temporal structure.

The **reconstruction-space PCA curve** may differ somewhat because timepoint matching occurs after projecting the compressed representation back into 700-node space. This is expected and is the appropriate curve for a direct comparison with the current MS-AA reconstruction pipeline.

## Full PCA vs MS-AA

Do not require AA to outperform PCA in absolute accuracy. PCA explicitly finds variance-maximizing linear directions and can act as a denoising transform.

The relevant question is whether AA obtains substantial stimulus-locked information while producing a representation whose informative elements have a distinct geometric and network interpretation.

## PCA-equivalent dimensionality

The `pca_K_min_reaching` column is especially useful.

For an AA configuration with \(m=5\), for example:

- `pca_K_min_reaching = 20` means five selected archetypes preserved at least as much timepoint-identification information as the monotonic PCA curve first achieved with 20 PCs.
- `pca_K_min_reaching = 5` means AA and PCA are similarly compact for that configuration.
- `NaN` means the AA configuration exceeded every PCA value tested.

This should be interpreted descriptively until top-\(m\) AA ranking is made independent/held-out, because the current AA top-\(m\) subsets were selected using their decoding performance.

## Why this comparison is cleaner

PCA has one conventional node-space compression axis, \(K=1,\ldots,700\). Its leading components are nested.

MS-AA solutions are not nested in the same way: changing parent model order \(K\) changes the archetypal geometry itself. Therefore a giant PCA \(K\times m\) landscape is neither necessary nor especially meaningful.

The clean comparison is instead:

1. conventional PCA information-retention curve;
2. full spatial and temporal AA curves;
3. sparse AA points relative to that PCA curve;
4. PCA-equivalent dimensionality of selected AA subsets;
5. network interpretation of the informative AA archetypes.
